# Nanozyme-ROS XGBoost Reproducibility Notebook


## 0. Environment and version check


In [ ]:
# 如果缺库，先在终端或此处安装，例如：
# !pip install xgboost shap scikit-learn pandas numpy matplotlib seaborn

import sys, platform
import numpy as np, pandas as pd
import sklearn
import xgboost
import seaborn as sns
import matplotlib
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('scikit-learn:', sklearn.__version__)
print('xgboost:', xgboost.__version__)
print('seaborn:', sns.__version__)
print('matplotlib:', matplotlib.__version__)


## 1. Reproducibility settings


In [ ]:
import os, random, numpy as np

SEED = 130  # 你可以改这个种子，但保持固定便于复现实验
random.seed(SEED)
np.random.seed(SEED)

# XGBoost 里也会传 random_state=SEED
print('Fixed SEED =', SEED)


## 2. Dataset path

The notebook reads the frozen v1.0 curated dataset from the repository `data/` directory. The default path works when this notebook is run from the `notebooks/` folder.


In [ ]:
from pathlib import Path

# Default path for the manuscript-facing GitHub release.
# Run this notebook from the notebooks/ directory, or adjust DATA_PATH if needed.
DATA_PATH = Path("../data/crop_nanozyme_ros_dataset_v1.0.xlsx")

if not DATA_PATH.exists():
    alt_path = Path("data/crop_nanozyme_ros_dataset_v1.0.xlsx")
    if alt_path.exists():
        DATA_PATH = alt_path
    else:
        raise FileNotFoundError(
            "Dataset not found. Expected ../data/crop_nanozyme_ros_dataset_v1.0.xlsx "
            "when running from notebooks/, or data/crop_nanozyme_ros_dataset_v1.0.xlsx "
            "when running from the repository root."
        )

print("Using data file:", DATA_PATH.resolve())

if DATA_PATH.suffix.lower() in [".xlsx", ".xls"]:
    df = pd.read_excel(DATA_PATH)
elif DATA_PATH.suffix.lower() == ".csv":
    df = pd.read_csv(DATA_PATH)
else:
    raise ValueError("Only .xlsx, .xls, and .csv files are supported.")

print("Data shape:", df.shape)
df.head()


## 3. Column check and type inference


In [ ]:
# 简要查看列与缺失情况
display(pd.DataFrame({'dtype': df.dtypes, 'n_null': df.isnull().sum()}))

# 自动推断：最后一列为目标（回归值）。如与你的原代码不同，请自行修改。
target_col = df.columns[-1]
print("Target column ->", target_col)

# 选择特征列（去掉目标列）
X_df = df.drop(columns=[target_col]).copy()
y = df[target_col].values

# 简单的类别/数值划分（可根据实际调整）
cat_cols = [c for c in X_df.columns if X_df[c].dtype == 'object']
num_cols = [c for c in X_df.columns if c not in cat_cols]

print("Categorical columns:", cat_cols)
print("Numeric columns:", num_cols)


## 4. Preprocessing and modelling (OneHotEncoder + XGBRegressor)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

# OneHotEncoder: 稀疏输出设为 False 便于与 pandas 交互；handle_unknown='ignore' 避免新类别报错
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(with_mean=False), num_cols),  # with_mean=False 以兼容稀疏矩阵
        ("cat", OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_cols),
    ],
    remainder='drop'
)

model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    reg_alpha=0.0,
    reg_lambda=1.0,
    n_jobs=-1
)

pipe = Pipeline(steps=[("pre", preprocess), ("xgb", model)])
pipe


## 5. Train/test split and model fitting


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_df, y, test_size=0.2, random_state=SEED)

pipe.fit(X_train, y_train)

print("训练完成。")


## 6. Evaluation metrics (RMSE / R2)


In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

pred_train = pipe.predict(X_train)
pred_test  = pipe.predict(X_test)

rmse_train = np.sqrt(mean_squared_error(y_train, pred_train))
rmse_test  = np.sqrt(mean_squared_error(y_test, pred_test))
r2_train   = r2_score(y_train, pred_train)
r2_test    = r2_score(y_test, pred_test)

print(f"Train RMSE: {rmse_train:.4f} | R2: {r2_train:.4f}")
print(f" Test RMSE: {rmse_test:.4f} | R2: {r2_test:.4f}")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import FancyBboxPatch

# 组装数据
df_train_plot = pd.DataFrame({"True": y_train, "Predicted": pred_train, "Split": "Train"})
df_test_plot  = pd.DataFrame({"True": y_test,  "Predicted": pred_test,  "Split": "Test"})
df_all = pd.concat([df_train_plot, df_test_plot], ignore_index=True)

# 轴范围（可按需要调整）
x_min = np.floor(df_all["True"].min()  - 0.2)
x_max = np.ceil( df_all["True"].max()  + 0.2)
y_min = np.floor(df_all["Predicted"].min() - 0.2)
y_max = np.ceil( df_all["Predicted"].max() + 0.2)

# JointGrid
g = sns.JointGrid(data=df_all, x="True", y="Predicted", height=7, xlim=(x_min, x_max), ylim=(y_min, y_max))

# 中心散点（分训练/测试）
g.ax_joint.scatter(df_train_plot["True"], df_train_plot["Predicted"],
                   s=36, alpha=0.85, edgecolor="white", linewidth=0.6, label="Train")
g.ax_joint.scatter(df_test_plot["True"], df_test_plot["Predicted"],
                   s=36, alpha=0.85, edgecolor="white", linewidth=0.6, label="Test")

# 两条回归线
sns.regplot(data=df_train_plot, x="True", y="Predicted", scatter=False, ax=g.ax_joint, label="Train Regression Line")
sns.regplot(data=df_test_plot,  x="True", y="Predicted", scatter=False, ax=g.ax_joint, label="Test Regression Line")

# x=y 参考线
lims = [min(x_min, y_min), max(x_max, y_max)]
g.ax_joint.plot(lims, lims, ls="--", linewidth=1, color="gray", alpha=0.8, label="x=y")
g.ax_joint.set_xlim(lims)
g.ax_joint.set_ylim(lims)


# 顶部：True 的直方图
sns.histplot(pd.to_numeric(df_train_plot["True"], errors="coerce").dropna(),
             bins=30, ax=g.ax_marg_x, alpha=0.5)
sns.histplot(pd.to_numeric(df_test_plot["True"], errors="coerce").dropna(),
             bins=30, ax=g.ax_marg_x, alpha=0.5)

# 右侧：Predicted 的直方图（放在 y 轴上）
sns.histplot(y=pd.to_numeric(df_train_plot["Predicted"], errors="coerce").dropna(),
             bins=30, ax=g.ax_marg_y, alpha=0.5)
sns.histplot(y=pd.to_numeric(df_test_plot["Predicted"], errors="coerce").dropna(),
             bins=30, ax=g.ax_marg_y, alpha=0.5)


# 文字与样式
g.ax_joint.set_xlabel("True Values")
g.ax_joint.set_ylabel("Predicted Values")
g.ax_joint.set_title("Model = XGBoost", loc="right", fontsize=10, pad=10)

# R^2 文本框
box_text = f"Train $R^2$ = {r2_train:.2f}\nTest  $R^2$ = {r2_test:.2f}"
bbox = dict(boxstyle="round,pad=0.4", fc="white", ec="black", alpha=0.8)
g.ax_joint.text(0.02, 0.02, box_text, transform=g.ax_joint.transAxes, fontsize=10, bbox=bbox)

# 图例
g.ax_joint.legend(loc="upper left", frameon=True)

plt.tight_layout()
plt.show()

# 同时保存文件
import pathlib
pathlib.Path("outputs").mkdir(exist_ok=True)
plt.savefig("outputs/pred_vs_true_joint.png", dpi=300, bbox_inches="tight")
print("Saved figure -> outputs/pred_vs_true_joint.png")


## 7. Cross-validation (optional)


In [ ]:
from sklearn.model_selection import cross_val_score
cv_score = cross_val_score(pipe, X_df, y, cv=5, scoring='r2', n_jobs=-1).mean()
print(f"5-Fold CV R2: {cv_score:.4f}")


## 8. Encoded feature names (`get_feature_names_out`)


In [ ]:
# 拿到 OHE 后的特征名：数值列 + 类别展开列
pre = pipe.named_steps['pre']
ohe = pre.named_transformers_['cat']

# 数值列本身名
num_names = num_cols.copy()

# 类别列展开的列名
cat_names = list(ohe.get_feature_names_out(cat_cols))

feature_names = num_names + cat_names
print("Encoded feature count:", len(feature_names))
feature_names[:20]


## 9. Export predictions and feature importance


In [ ]:
from pathlib import Path
import pandas as pd

out_dir = Path("./outputs")
out_dir.mkdir(exist_ok=True)

# 保存测试集预测
pred_df = pd.DataFrame({
    "True": y_test,
    "Predicted": pred_test
})
pred_path = out_dir / "predictions_xgb.xlsx"
pred_df.to_excel(pred_path, index=False)
print("Saved predictions ->", pred_path.resolve())

# 基于 booster 的特征重要性（注意：这与编码后的列名一一对应需要提取 pipeline 中的矩阵）
# 简化：这里用 X_train 经过预处理后的矩阵 + 模型 feature_importances_
from sklearn import set_config
set_config(transform_output="pandas")
X_train_enc = pre.fit_transform(X_train)  # 注意：为了拿到列名，这里重新 fit 一次 preprocess（与上面 pipe.fit 参数一致）
X_train_enc.columns = feature_names

booster_importance = pipe.named_steps['xgb'].feature_importances_
fi_df = pd.DataFrame({"feature": X_train_enc.columns, "importance": booster_importance}).sort_values("importance", ascending=False)
fi_path = out_dir / "feature_importance_xgb.xlsx"
fi_df.to_excel(fi_path, index=False)
print("Saved feature importance ->", fi_path.resolve())

fi_df.head(20)


## 10. SHAP model interpretation (optional)


In [ ]:
# 如果未安装：!pip install shap
try:
    import shap
    shap.initjs()
    # 使用一小部分样本做解释以节约时间
    X_sample = X_train.sample(n=min(200, len(X_train)), random_state=SEED)
    # 经过预处理
    X_sample_enc = pre.transform(X_sample)
    explainer = shap.TreeExplainer(pipe.named_steps['xgb'])
    shap_values = explainer.shap_values(X_sample_enc)
    # 总体特征重要性图（需要在 notebook 里显示）
    shap.summary_plot(shap_values, X_sample_enc, feature_names=feature_names, show=False)
except Exception as e:
    print("SHAP 可选步骤失败/未安装：", e)
